# TensorFlow Serving: REST, gRPC i wersjonowanie modeli

Notebook demonstruje model server do uruchomienia i utrzymania modelu. Model server pozwala wygodnie zarządzać wersjami modeli produkcyjnie.


In [ ]:
from pathlib import Path
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train[:10000], y_train[:10000], validation_split=0.1, epochs=2, batch_size=128)

export_path = Path("models/mnist/1")
export_path.mkdir(parents=True, exist_ok=True)
model.export(str(export_path))
print(f"SavedModel exported to: {export_path.resolve()}")


## Uruchomienie TensorFlow Serving przez Docker

W terminalu, z katalogu projektu:

```bash
docker run -p 8501:8501 -p 8500:8500 \
  --name tfserving_mnist \
  --rm \
  -v "$PWD/models/mnist:/models/mnist" \
  -e MODEL_NAME=mnist \
  tensorflow/serving
```

Port `8501` to REST, a `8500` to gRPC.


In [ ]:
import json
import requests

# Przykład REST. Wymaga uruchomionego TensorFlow Serving.
image = x_test[0].tolist()
payload = {"instances": [image]}

# response = requests.post("http://localhost:8501/v1/models/mnist:predict", json=payload)
# print(response.json())

print("Odkomentuj request po uruchomieniu TensorFlow Serving.")


## Wersjonowanie modelu

Aby dodać nową wersję, wytrenuj model ponownie i zapisz go do katalogu:

```text
models/mnist/2/
```

TensorFlow Serving może obsługiwać nowe wersje modeli bez zmiany kodu klienta. To pokazuje jedną z najważniejszych zalet model servera: zarządzanie cyklem życia modelu.


## gRPC — szkic klienta

W praktyce gRPC wymaga zależności `tensorflow-serving-api`.

```bash
pip install tensorflow-serving-api grpcio
```


In [ ]:
# Szkic klienta gRPC - do omówienia podczas wykładu
# import grpc
# import numpy as np
# import tensorflow as tf
# from tensorflow_serving.apis import prediction_service_pb2_grpc, predict_pb2
#
# channel = grpc.insecure_channel("localhost:8500")
# stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)
# request = predict_pb2.PredictRequest()
# request.model_spec.name = "mnist"
# request.model_spec.signature_name = "serving_default"
# request.inputs["input_1"].CopyFrom(tf.make_tensor_proto([x_test[0]], dtype=tf.float32))
# result = stub.Predict(request, timeout=10.0)
# print(result)

print("gRPC zwykle jest bardziej wydajne niż REST przy komunikacji wewnętrznej między usługami, ale jest mniej wygodne do ręcznego debugowania.")


## Podsumowanie

- Flask/FastAPI są świetne na start.
- TensorFlow Serving jest wyspecjalizowanym model serverem.
- Najważniejsza różnica: wersje modeli, REST/gRPC, canary/A/B testing i mniejszy narzut.
- W produkcji deployment modelu to proces, a nie tylko jeden plik `.h5` albo `.keras`.
